<a href="https://colab.research.google.com/github/HARSHINI-0523/Amazon_ML_Challenge_2K25/blob/main/Amazon_ML_Challenge_2K25_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# STAGE 1: DATA SETUP & ENVIRONMENT
# ==============================================================================
print("🚀 Stage 1: Setting up the environment...")

!pip install xgboost lightgbm optuna -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
from zipfile import ZipFile
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import make_scorer
import torch
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/ML_Challenge_2025/"
DATA_PATH = os.path.join(BASE_PATH, "dataset/")
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs/")
SRC_PATH = os.path.join(BASE_PATH, "src/")
IMAGE_PATH = os.path.join(DATA_PATH, "images/")
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(IMAGE_PATH, exist_ok=True)
os.makedirs(SRC_PATH, exist_ok=True)
try:
    train_df = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'))
    test_df = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'))
    print("✅ Data loaded successfully.")
    print("Train data shape:", train_df.shape)
    print("Test data shape:", test_df.shape)
    print("\nFirst 5 rows of training data:")
    display(train_df.head())
except FileNotFoundError:
    print("❌ Error: 'train.csv' or 'test.csv' not found in the specified Google Drive path.")
    print(f"Please upload your data to: {DATA_PATH}")

def smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error (SMAPE)"""
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    return np.mean(numerator / denominator) * 100

smape_scorer = make_scorer(smape, greater_is_better=False)

def clean_text(text):
    """Basic text cleaning."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("✅ Stage 1 Complete.\n" + "="*60 + "\n")

🚀 Stage 1: Setting up the environment...
Mounted at /content/drive
✅ Data loaded successfully.
Train data shape: (75000, 4)
Test data shape: (75000, 3)

First 5 rows of training data:


,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49


✅ Stage 1 Complete.



In [ ]:
# ==============================================================================
# STAGE 2: PROTOTYPE ON DATA
# ==============================================================================
print("🚀 Stage 2: Prototyping on a small data subset...")

train_small_df = train_df.sample(frac=0.1, random_state=42)
print(f"Working with {len(train_small_df)} samples for prototyping.")

train_small_df['cleaned_content'] = train_small_df['catalog_content'].apply(clean_text)

# 1. Feature Engineering (TF-IDF)
tfidf_vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
X_text_small = tfidf_vectorizer.fit_transform(train_small_df['cleaned_content'])
y_small = train_small_df['price']

# 2. Train-Validation Split
X_train, X_val, y_train, y_val = train_test_split(X_text_small, y_small, test_size=0.2, random_state=42)

# 3. Model Training
xgb_proto = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42)
xgb_proto.fit(X_train, y_train)

# 4. Evaluation
preds = xgb_proto.predict(X_val)
proto_smape = smape(y_val, preds)
print(f"✅ Prototyping model trained. Validation SMAPE: {proto_smape:.4f}")
print("✅ End-to-end pipeline is working on a small dataset.")
print("✅ Stage 2 Complete.\n" + "="*60 + "\n")

🚀 Stage 2: Prototyping on a small data subset...
Working with 7500 samples for prototyping.
✅ Prototyping model trained. Validation SMAPE: 67.2454
✅ End-to-end pipeline is working on a small dataset.
✅ Stage 2 Complete.



In [ ]:
# ==============================================================================
#  STAGE 3: TEXT-ONLY MODEL (BASELINE ON FULL DATA)
# ==============================================================================
print("🚀 Stage 3: Building a strong text-only baseline on full data...")

train_df['cleaned_content'] = train_df['catalog_content'].apply(clean_text)

# 1. Feature Engineering
tfidf_full = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words='english'
)
X_text_full = tfidf_full.fit_transform(train_df['cleaned_content'])
y_full = train_df['price']

# 2. Train-Validation Split
X_train_full, X_val_full, y_train_full, y_val_full = train_test_split(
    X_text_full, y_full, test_size=0.2, random_state=42
)

# 3. Model Training
lgbm_baseline = lgb.LGBMRegressor(random_state=42, n_estimators=500)
lgbm_baseline.fit(X_train_full, y_train_full,
                  eval_set=[(X_val_full, y_val_full)],
                  eval_metric='l1',
                  callbacks=[lgb.early_stopping(10)])

# 4. Evaluation
preds_baseline = lgbm_baseline.predict(X_val_full)
baseline_smape = smape(y_val_full, preds_baseline)
print(f"✅ Text-only baseline model trained. Validation SMAPE: {baseline_smape:.4f}")
print("✅ Stage 3 Complete.\n" + "="*60 + "\n")

🚀 Stage 3: Building a strong text-only baseline on full data...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 27.924750 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1080139
[LightGBM] [Info] Number of data points in the train set: 60000, number of used features: 19586
[LightGBM] [Info] Start training from score 23.598634
Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[171]	valid_0's l1: 13.4948	valid_0's l2: 1071.28


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ Text-only baseline model trained. Validation SMAPE: 62.0210
✅ Stage 3 Complete.



In [ ]:
# ==============================================================================
# STAGE 4: BATCH-PROCESSING FOR IMAGE FEATURES (MEMORY EFFICIENT)
# ==============================================================================
import requests
import shutil

print("🚀 Stage 4: Extracting features in batches to save disk space...")
BATCH_SIZE = 500
IMAGE_FEATURES_PATH = os.path.join(OUTPUT_PATH, "images/")
TEMP_IMAGE_BATCH_DIR = os.path.join(DATA_PATH, "temp_image_batch/")

os.makedirs(IMAGE_FEATURES_PATH, exist_ok=True)
num_batches = int(np.ceil(len(train_df) / BATCH_SIZE))

for i in tqdm(range(num_batches), desc="Processing Batches"):
    start_idx = i * BATCH_SIZE
    end_idx = start_idx + BATCH_SIZE
    batch_df = train_df.iloc[start_idx:end_idx]

    # --- 1. DOWNLOAD BATCH ---
    os.makedirs(TEMP_IMAGE_BATCH_DIR, exist_ok=True)
    print(f"\nBatch {i+1}/{num_batches}: Downloading {len(batch_df)} images...")
    for _, row in batch_df.iterrows():
        image_url = row['image_link']
        sample_id = row['sample_id']
        image_filepath = os.path.join(TEMP_IMAGE_BATCH_DIR, f"{sample_id}.jpg")

        if not os.path.exists(image_filepath):
            try:
                response = requests.get(image_url, timeout=10)
                if response.status_code == 200:
                    with open(image_filepath, 'wb') as f:
                        f.write(response.content)
            except Exception as e:
                print(f"Skipping download for {sample_id}: {e}")

    # --- 2. PROCESS BATCH ---
    print(f"Batch {i+1}/{num_batches}: Extracting features...")
    for _, row in batch_df.iterrows():
        sample_id = row['sample_id']
        image_file = os.path.join(TEMP_IMAGE_BATCH_DIR, f"{sample_id}.jpg")
        output_feature_file = os.path.join(IMAGE_FEATURES_PATH, f"{sample_id}.npy")
        if not os.path.exists(output_feature_file):
            if os.path.exists(image_file):
                features = extract_image_features(image_file)
                np.save(output_feature_file, features)

    # --- 3. DELETE BATCH ---
    print(f"Batch {i+1}/{num_batches}: Cleaning up image files...")
    shutil.rmtree(TEMP_IMAGE_BATCH_DIR)

print("\n✅ All batches processed. Image feature extraction complete.")
print("✅ Stage 4 Complete.\n" + "="*60 + "\n")

🚀 Stage 4: Extracting features in batches to save disk space...


Processing Batches:   0%|          | 0/150 [00:00<?, ?it/s]


Batch 1/150: Downloading 500 images...
Batch 1/150: Extracting features...
Batch 1/150: Cleaning up image files...


Processing Batches:   1%|          | 1/150 [00:56<2:19:40, 56.24s/it]


Batch 2/150: Downloading 500 images...
Batch 2/150: Extracting features...
Batch 2/150: Cleaning up image files...


...


Batch 149/150: Downloading 500 images...
Batch 149/150: Extracting features...
Batch 149/150: Cleaning up image files...


Processing Batches:  99%|█████████▉| 149/150 [2:35:19<01:01, 61.22s/it]


Batch 150/150: Downloading 500 images...
Batch 150/150: Extracting features...
Batch 150/150: Cleaning up image files...


Processing Batches: 100%|██████████| 150/150 [2:36:20<00:00, 62.53s/it]


✅ All batches processed. Image feature extraction complete.
✅ Stage 4 Complete.



In [ ]:
IMAGE_FEATURES_PATH = os.path.join(OUTPUT_PATH, "image_features/")
os.makedirs(IMAGE_FEATURES_PATH, exist_ok=True)

In [ ]:
# ==============================================================================
# STAGE 5: FUSION MODEL (TEXT + IMAGE)
# ==============================================================================
print("🚀 Stage 5: Building a fusion model with real text and image features...")
def load_image_features(df, path):
    """Loads saved .npy feature files for each sample_id in the dataframe."""
    features_list = []
    print(f"Loading image features from: {path}")
    for sample_id in tqdm(df['sample_id'], desc="Loading .npy files"):
        feature_file = os.path.join(path, f"{sample_id}.npy")
        if os.path.exists(feature_file):
            features = np.load(feature_file)
            features_list.append(features)
        else:
            features_list.append(np.zeros(2048))
    return np.array(features_list)
image_features_full = load_image_features(train_df, IMAGE_FEATURES_PATH)
from scipy.sparse import hstack
X_fusion = hstack([X_text_full, image_features_full]).tocsr()
y_full = train_df['price']
print(f"Fusion successful. New feature matrix shape: {X_fusion.shape}")
X_train_fus, X_val_fus, y_train_fus, y_val_fus = train_test_split(
    X_fusion, y_full, test_size=0.2, random_state=42
)
lgbm_fusion = lgb.LGBMRegressor(random_state=42, n_estimators=500, n_jobs=-1)
lgbm_fusion.fit(X_train_fus, y_train_fus,
                eval_set=[(X_val_fus, y_val_fus)],
                eval_metric='l1',
                callbacks=[lgb.early_stopping(15)])
preds_fusion = lgbm_fusion.predict(X_val_fus)
fusion_smape = smape(y_val_fus, preds_fusion)
print(f"\n✅ Text-only baseline SMAPE: {baseline_smape:.4f}")
print(f"✅ Fusion model SMAPE: {fusion_smape:.4f}")
print(f"📈 Performance boost from images: {baseline_smape - fusion_smape:.4f}")
print("✅ Stage 5 Complete.\n" + "="*60 + "\n")

🚀 Stage 5: Building a fusion model with real text and image features...
Loading image features from: /content/drive/MyDrive/ML_Challenge_2025/outputs/image_features/


Loading .npy files: 100%|██████████| 75000/75000 [03:33<00:00, 350.57it/s] 


Fusion successful. New feature matrix shape: (75000, 22048)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 21.138993 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1211087
[LightGBM] [Info] Number of data points in the train set: 60000, number of used features: 21631
[LightGBM] [Info] Start training from score 23.598634
Training until validation scores don't improve for 15 rounds
Early stopping, best iteration is:
[225]	valid_0's l1: 13.3845	valid_0's l2: 1067.27


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



✅ Text-only baseline SMAPE: 62.0210
✅ Fusion model SMAPE: 61.4702
📈 Performance boost from images: 0.5508
✅ Stage 5 Complete.



In [ ]:
# ==============================================================================
# 🧪 STAGE 6: HYPERPARAMETER TUNING & ENSEMBLING
# ==============================================================================
print("🚀 Stage 6: Tuning hyperparameters and ensembling models...")
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
def objective(trial):
    params = {
        'objective': 'regression_l1',
        'metric': 'mae',
        'n_estimators': 2000,
        'random_state': 42,
        'n_jobs': -1,
        'learning_rate': trial.suggest_float('learning_rate', 1e-2, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train_fus, y_train_fus,
              eval_set=[(X_val_fus, y_val_fus)],
              eval_metric='l1',
              callbacks=[lgb.early_stopping(20)])
    preds = model.predict(X_val_fus)
    return smape(y_val_fus, preds)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30)
print(f"\nBest trial SMAPE: {study.best_value:.4f}")
print("Best hyperparameters:", study.best_params)
best_lgbm_params = study.best_params
# 1. Train the tuned LightGBM model
lgbm_tuned = lgb.LGBMRegressor(**best_lgbm_params, random_state=42, n_estimators=2000)
lgbm_tuned.fit(X_train_fus, y_train_fus,
               eval_set=[(X_val_fus, y_val_fus)],
               callbacks=[lgb.early_stopping(20)])
preds_lgbm_tuned = lgbm_tuned.predict(X_val_fus)
print(f"\nTuned LightGBM SMAPE: {smape(y_val_fus, preds_lgbm_tuned):.4f}")
# 2. Train a solid XGBoost model
xgb_tuned = xgb.XGBRegressor(objective='reg:squarederror',
                             n_estimators=1000,
                             learning_rate=0.05,
                             random_state=42,
                             n_jobs=-1,
                             tree_method='gpu_hist') # Use GPU for XGBoost
xgb_tuned.fit(X_train_fus, y_train_fus,
              eval_set=[(X_val_fus, y_val_fus)],
              early_stopping_rounds=20,
              verbose=False)
preds_xgb_tuned = xgb_tuned.predict(X_val_fus)
print(f"XGBoost SMAPE: {smape(y_val_fus, preds_xgb_tuned):.4f}")
# 3. Simple Averaging Ensemble
ensemble_preds = (preds_lgbm_tuned + preds_xgb_tuned) / 2
ensemble_smape = smape(y_val_fus, ensemble_preds)

print(f"\n✅ Final Ensemble SMAPE on Validation Set: {ensemble_smape:.4f}")
print("✅ Stage 6 Complete.\n" + "="*60 + "\n")

🚀 Stage 6: Tuning hyperparameters and ensembling models...
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] feature_fraction is set=0.9922098242143459, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9922098242143459
[LightGBM] [Warning] bagging_fraction is set=0.9901781999843211, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9901781999843211
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] feature_fraction is set=0.9922098242143459, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9922098242143459
[LightGBM] [Warning] bagging_fraction is set=0.9901781999843211, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9901781999843211
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 9.993469 seconds.
You can set `force_

Exception ignored on calling ctypes callback function: <function _log_callback at 0x7cd4837c56c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
    
KeyboardInterrupt: 


No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


Exception ignored on calling ctypes callback function: <function _log_callback at 0x7cd4837c56c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
    
KeyboardInterrupt: 


No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

...

No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


Exception ignored on calling ctypes callback function: <function _log_callback at 0x7cd4837c56c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
    
KeyboardInterrupt: 


No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


Exception ignored on calling ctypes callback function: <function _log_callback at 0x7cd4837c56c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
    
KeyboardInterrupt: 


No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


In [ ]:
# ==============================================================================
# STAGE - 7 : FINAL TRAINING & SUBMISSION
# ==============================================================================
print("🚀 Executing Plan : Generating a submission...")
print("Training final LightGBM model on all 75k text samples...")
final_text_lgbm = lgb.LGBMRegressor(random_state=42, n_estimators=300)
final_text_lgbm.fit(X_text_full, y_full)
print("Preparing test data (text features only)...")
test_df['cleaned_content'] = test_df['catalog_content'].apply(clean_text)
X_text_test = tfidf_full.transform(test_df['cleaned_content'])
print("Generating predictions...")
text_only_predictions = final_text_lgbm.predict(X_text_test)
submission_df_text_only = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': text_only_predictions
})
submission_df_text_only['price'] = submission_df_text_only['price'].clip(lower=0)
submission_path_text_only = os.path.join(OUTPUT_PATH, 'submission_text_only.csv')
submission_df_text_only.to_csv(submission_path_text_only, index=False)

print(f"\submission file created at: {submission_output}")
print("This file is ready to be submitted to the hackathon!")
display(submission_df_text_only.head())

<>:30: SyntaxWarning: invalid escape sequence '\s'
<>:30: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-1825368586.py:30: SyntaxWarning: invalid escape sequence '\s'
  print(f"\submission file created at: {submission_path_text_only}")


🚀 Executing Plan : Generating a submission...
Training final LightGBM model on all 75k text samples...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 28.961023 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1258236
[LightGBM] [Info] Number of data points in the train set: 75000, number of used features: 19766
[LightGBM] [Info] Start training from score 23.647654
Preparing test data (text features only)...
Generating predictions...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


\submission file created at: /content/drive/MyDrive/ML_Challenge_2025/outputs/submission_text_only.csv
This file is ready to be submitted to the hackathon!


,sample_id,price
0,100179,19.350114
1,245611,28.237350
2,146263,30.000128
3,95658,17.932504
4,36806,38.738847
